# E2 — Siamese đa pha · train trên Kaggle

Một encoder DenseNet121-3D **dùng chung**, chạy riêng cho từng thì, rồi hợp nhất bằng
phase-attention. Thay cho early-concat v0 của E0/E1.

**Mốc đối chiếu** (fold 1, val 82 ca, WORKLOG S-060):

| | macro-F1 |
|---|---|
| E0 early-concat + cache `fixed_mm` | 0,4244 |
| E1 early-concat + cache `lesion_tight` | 0,5740 |
| baseline official (test-104) | 0,6083 |

**Bốn cổng chặn, chạy đúng thứ tự.** Mỗi cổng tồn tại vì một lỗi đã xảy ra thật:

0. `pytest` — model này **chưa từng chạy forward pass**, local không có torch (S-061).
1. Shape + trọng số attention — bắt lệch hợp đồng trước khi tốn GPU.
2. Overfit 8 mẫu — S-039 mất một run vì norm không học nổi.
3. Đo thời gian — Siamese chạy backbone 8 lượt, FLOPs ~8×.

> ⚠️ **`input_downsample: 2` là biến gây nhiễu.** E2 ở đây là *Siamese ở nửa độ phân
> giải* so với E1 là *early-concat ở đủ độ phân giải*. E2 thắng thì kết luận mạnh
> (thắng dù bị thiệt). **E2 thua thì không kết luận được**, phải chạy thêm E1 với
> cùng `input_downsample` làm đối chứng.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"
ON_KAGGLE = Path("/kaggle/input").exists()

EXPERIMENT = "E2_siamese"
CONFIG_NAME = "e2_siamese.yaml"

if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    subprocess.run(["rm", "-rf", str(REPO)], check=False)
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    sys.path.insert(0, str(REPO))
    os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(REPO))

# Xoá module src.* đã nạp từ lần chạy trước. Clone lại code KHÔNG tự làm điều này:
# Python giữ nguyên bản đã import trong sys.modules (WORKLOG S-035).
for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

if ON_KAGGLE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True
    )

from src.utils.io import repo_root  # noqa: E402

print("code đang dùng:", repo_root())
assert repo_root() == REPO.resolve(), (
    f"module src/ đang nạp từ {repo_root()} chứ không phải {REPO}. Restart kernel."
)

## Cổng 0 ⚠️ — model chưa từng chạy forward pass

`src/models/siamese_fusion.py` được viết trên máy **không có torch**, nên 6 test của nó
đang skip ở local. Đây là lần đầu chúng chạy thật. Mất vài giây, rẻ hơn 4 giờ GPU rất nhiều.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_models.py", "-q"],
    cwd=REPO, capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("test model KHÔNG qua — đừng train. Gửi nguyên đoạn trên.")
print("Cổng 0 qua.")

## 1. Chọn cache — **tường minh, không tự dò**

Nếu mount cả cache `fixed_mm` lẫn `lesion_tight` thì `find_cache_dir` chọn bừa một cái,
và bạn sẽ không biết mình vừa train trên dữ liệu nào. E2 phải dùng **`lesion_tight`**,
giống E1, nếu không thì không so được.

In [ ]:
import json
import pathlib

print("Các cache đang mount:")
for p in sorted(pathlib.Path("/kaggle/input").rglob("cache_meta.json")):
    meta = json.loads(p.read_text(encoding="utf-8"))
    print(f"  {p.parent}")
    print(f"     crop_mode: {meta.get('crop_mode', '(thiếu khoá — cache v0 fixed_mm)')}"
          f" | target_size: {meta.get('target_size')}"
          f" | .npz: {len(list(p.parent.glob('*.npz')))}")

In [ ]:
# >>> ĐIỀN đường dẫn cache lesion_tight lấy từ cell trên <<<
CACHE_DIR_STR = "/kaggle/input/..../cache_lesion_tight"

import json
import os
from pathlib import Path

meta_path = Path(CACHE_DIR_STR) / "cache_meta.json"
assert meta_path.is_file(), f"không thấy {meta_path} — kiểm lại đường dẫn ở cell trên"
meta = json.loads(meta_path.read_text(encoding="utf-8"))
assert meta.get("crop_mode") == "lesion_tight", (
    f"cache này là {meta.get('crop_mode')!r}, E2 phải dùng 'lesion_tight' để so được với E1"
)
os.environ["LLDMMRI_CACHE_DIR"] = CACHE_DIR_STR

from src.utils.io import load_yaml, repo_root, resolve_cache_dir

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
CACHE_DIR = resolve_cache_dir(CFG)
SPLITS_DIR = repo_root() / CFG.get("splits_dir", "splits")
FOLD = CFG["fold"]

n_npz = len(list(CACHE_DIR.glob("*.npz")))
print("cache   :", CACHE_DIR, f"({n_npz} file .npz, cần 498)")
print("config  :", CFG_PATH.name)
print("model   :", CFG["model"])
print("output  :", os.environ.get("LLDMMRI_OUTPUT_DIR"))
assert n_npz == 498, f"chỉ thấy {n_npz}/498 file — dừng lại kiểm tra"

## Cổng 1 — shape và trọng số attention

Kiểm hợp đồng `[B, 8, X, Y, Z] → [B, 7]`, và kiểm `last_phase_weights` là một phân bố
hợp lệ trên 8 thì. Trọng số này là **đầu ra khoa học** dùng cho ablation
phase-importance ở W4, không phải chi tiết nội bộ.

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.data.dataset import build_fold_datasets
from src.data.taxonomy import SHORT_NAMES
from src.models import build_model, count_parameters

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

train_ds, val_ds = build_fold_datasets(CACHE_DIR, FOLD, splits_dir=SPLITS_DIR)
print(f"fold {FOLD}: train={len(train_ds)} val={len(val_ds)} (tổng phải = 394)")

model = build_model(CFG["model"]).to(DEVICE)
print(f"tham số: {count_parameters(model):,}")

batch = next(iter(DataLoader(train_ds, batch_size=2)))
images = batch["image"].to(DEVICE)
model.eval()
with torch.no_grad():
    logits = model(images)

print(f"\nvào  {tuple(images.shape)}  ->  ra {tuple(logits.shape)}")
assert logits.shape == (images.shape[0], 7), "hợp đồng shape sai"
assert torch.isfinite(logits).all(), "logits có NaN/Inf"

weights = model.last_phase_weights
if weights is not None:
    assert torch.allclose(weights.sum(1), torch.ones(len(weights), device=DEVICE), atol=1e-4)
    phases = [p["name"] for p in load_yaml(REPO / "configs" / "data.yaml")["phases"]]
    print("\ntrọng số attention (chưa train, nên phải gần đều 1/8 = 0.125):")
    for name, w in zip(phases, weights[0].tolist()):
        print(f"   {name:>10}: {w:.4f}")
print("\nCổng 1 qua.")

## Cổng 2 ⚠️ — model có học nổi 8 mẫu không

Nếu không overfit nổi 8 mẫu thì có lỗi cấu trúc, và 4 giờ GPU sẽ vô ích.
S-039 đã mất một run vì đúng chuyện này (InstanceNorm + global average pooling).

In [ ]:
import gc

from src.train.sanity import overfit_check, verdict
from src.utils.seed import set_seed

set_seed(CFG["seed"])
probe = build_model(CFG["model"])
result = overfit_check(train_ds, probe, DEVICE, n_samples=8, passes=40)

print(f"loss đầu {result['loss_start']:.3f} -> cuối {result['loss_end']:.3f} "
      f"| acc {result['accuracy_end']:.2f}")
print(f"kết luận: {verdict(result)}")
print("(8 mẫu; loss của đoán ngẫu nhiên = ln 7 = 1.946)")

del probe
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

if verdict(result) == "SẬP":
    raise RuntimeError(
        "Model không học nổi 8 mẫu. Đừng train. Nghi ngờ đầu tiên: "
        "phase_embedding khởi tạo 0 cộng vào đặc trưng, hoặc norm=batch với batch nhỏ."
    )
print("Cổng 2 qua.")

## Cổng 3 ⚠️ — đo thời gian

Siamese chạy backbone **8 lượt** nên FLOPs ~8×. `input_downsample: 2` cắt voxel đi 8 lần
để bù lại, nhưng đó là **tính toán, chưa đo**. Cell này đo thật.

In [ ]:
import gc
import time

from src.train.loop import make_amp_scaler, run_epoch
from src.train.run import build_loaders, build_param_groups, build_scheduler
from src.utils.seed import set_seed

BUDGET_HOURS_PER_FOLD = 6.0
PROBE_EPOCHS = 2

TCFG, DCFG = CFG["train"], CFG["data"]
set_seed(CFG["seed"])

probe_train_loader, probe_val_loader, probe_labels = build_loaders(CFG, FOLD)
probe_model = build_model(CFG["model"]).to(DEVICE)
probe_opt = torch.optim.AdamW(
    build_param_groups(probe_model, float(TCFG["weight_decay"])), lr=float(TCFG["lr"])
)
probe_sched = build_scheduler(probe_opt, TCFG, int(TCFG["epochs"]))
probe_amp = bool(TCFG.get("amp", True)) and DEVICE.type == "cuda"
probe_scaler = make_amp_scaler(probe_amp)
criterion = torch.nn.CrossEntropyLoss()
accum = int(TCFG["accum_steps"])

print(f"batch hiệu dụng {int(DCFG['batch_size']) * accum} · "
      f"~{len(probe_labels) // (int(DCFG['batch_size']) * accum)} bước/epoch")

timings = []
for i in range(PROBE_EPOCHS):
    t0 = time.time()
    tr = run_epoch(probe_model, probe_train_loader, DEVICE, criterion,
                   optimizer=probe_opt, scaler=probe_scaler, accum_steps=accum, amp=probe_amp)
    va = run_epoch(probe_model, probe_val_loader, DEVICE, criterion, amp=probe_amp)
    probe_sched.step()
    timings.append(time.time() - t0)
    print(f"epoch thử {i + 1}: {timings[-1]:.1f}s | train {tr['loss']:.4f} | val {va['loss']:.4f}")

per_epoch = timings[-1]   # epoch đầu gánh chi phí khởi động worker
hours_one = per_epoch * int(TCFG["epochs"]) / 3600
print(f"\n~{per_epoch:.1f}s/epoch × {TCFG['epochs']} epoch = **{hours_one:.2f} giờ/fold**")
print(f"(E1 mất 4.09h/fold — nếu con số này gần bằng thì input_downsample đã bù đúng)")

del probe_model, probe_opt, probe_sched, probe_scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

if hours_one > BUDGET_HOURS_PER_FOLD:
    raise RuntimeError(
        f"{hours_one:.2f} giờ/fold, vượt ngân sách {BUDGET_HOURS_PER_FOLD} giờ.\n"
        "Cách xử lý, theo thứ tự: (1) tăng input_downsample lên 3 — nhưng ghi WORKLOG "
        "vì nó làm biến gây nhiễu nặng thêm; (2) giảm embed_dim; (3) giảm epochs và "
        "ghi rõ là đã lệch recipe official."
    )
print(f"Cổng 3 qua — trong ngân sách {BUDGET_HOURS_PER_FOLD} giờ/fold.")

## 2. Train fold 1

Chạy lại đúng cell này nếu session bị ngắt — `resume: true` sẽ tiếp tục từ `last.pt`.

In [ ]:
from src.train.run import train

result = train(CFG_PATH, fold_override=FOLD)
print(result)

## 3. Kết quả và đối chiếu E0/E1

In [ ]:
import json

import numpy as np

from src.train.run import run_dir

RUN_DIR = run_dir(CFG, FOLD)
print("run dir:", RUN_DIR)
best = json.loads((RUN_DIR / "metrics_best.json").read_text(encoding="utf-8"))

print(f"\nfold {best['fold']} · epoch {best['epoch']} · seed {best['seed']}")
for key in ("macro_f1", "balanced_accuracy", "accuracy", "cohen_kappa"):
    print(f"  {key:>18}: {best[key]:.4f}")

print("\nF1 từng lớp:")
for k, f1 in enumerate(best["per_class_f1"]):
    print(f"  {SHORT_NAMES[k]:>7}: {f1:.3f}")

print("\nMa trận nhầm lẫn (hàng = thật, cột = đoán):")
matrix = np.array(best["confusion_matrix"])
print("        " + "".join(f"{SHORT_NAMES[k]:>8}" for k in sorted(SHORT_NAMES)))
for k, row in enumerate(matrix):
    print(f"{SHORT_NAMES[k]:>7} " + "".join(f"{v:>8d}" for v in row))

print("\n--- ĐỐI CHIẾU (cùng fold 1, cùng 82 ca val) ---")
print(f"  E0  early-concat + fixed_mm     : 0.4244")
print(f"  E1  early-concat + lesion_tight : 0.5740")
print(f"  E2  siamese      + lesion_tight : {best['macro_f1']:.4f}   <- lần này")
print(f"  baseline official (test-104)    : 0.6083")
print(f"\n  E2 - E1 = {best['macro_f1'] - 0.5740:+.4f}")

### Calibration và selective

Xác suất val đã được lưu ở `val_probs_best.npz`, nên phần này chạy trên CPU và
lặp lại được bất cứ lúc nào mà không cần GPU.

`T` được **cross-fit 5 phần** (học trên 4/5, áp lên 1/5). Fit thẳng trên chính tập
đánh giá cho ECE tốt giả tạo ~44% (WORKLOG S-060).

In [ ]:
import numpy as np

from src.eval import calibration as C
from src.eval import selective as S
from src.eval.bootstrap import bootstrap_metric
from src.eval.metrics import macro_f1

d = np.load(RUN_DIR / "val_probs_best.npz", allow_pickle=True)
probs = d["probs"].astype(np.float64)
probs /= probs.sum(1, keepdims=True)
labels = d["labels"]
pred = probs.argmax(1)
conf = probs.max(1)
correct = (pred == labels).astype(int)


def crossfit_T(p, y, k=5, seed=1337):
    idx = np.random.default_rng(seed).permutation(len(y))
    out = np.empty_like(p)
    temps = []
    for i in range(k):
        te = idx[i::k]
        tr = np.setdiff1d(idx, te)
        t = C.fit_temperature(p[tr], y[tr])
        temps.append(t)
        out[te] = C.apply_temperature(p[te], t)
    return out, np.array(temps)


cal, temps = crossfit_T(probs, labels)
ci = bootstrap_metric(labels, pred, macro_f1, n_resamples=4000)

print(f"macro-F1 {ci['point']:.4f} [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")
print(f"ECE  {C.expected_calibration_error(probs, labels):.4f}"
      f" -> {C.expected_calibration_error(cal, labels):.4f} sau temperature")
print(f"NLL  {C.negative_log_likelihood(probs, labels):.4f}"
      f" -> {C.negative_log_likelihood(cal, labels):.4f}   (đoán mò = {np.log(7):.4f})")
print(f"T    {temps.mean():.3f} ± {temps.std():.3f}")
print(f"AURC {S.aurc(correct, conf):.4f}   (thấp là tốt)")

print("\naccuracy theo coverage:")
for cov in (1.0, 0.9, 0.8, 0.7):
    print(f"   {cov:4.0%}  {S.selective_accuracy(correct, conf, cov):.4f}")

print("\n⚠️ KHÔNG báo macro-F1@coverage trên một fold: ở coverage thấp lớp hiếm tụt")
print("   xuống 1-2 ca và con số vô nghĩa (WORKLOG S-060). Đợi gộp out-of-fold 5 fold.")

print("\n--- ĐỐI CHIẾU calibration ---")
print("  E0: ECE 0.3218 -> 0.1455 | NLL 2.7172 -> 1.7251 | AURC 0.5395")
print("  E1: ECE 0.2935 -> 0.2505 | NLL 3.3182 -> 1.5205 | AURC 0.2753")

### Trọng số attention từng thì

Đây là thứ early-concat không cho được: model tự nói nó dựa vào thì nào.
Kỳ vọng theo LI-RADS là **arterial và venous** nổi bật (arterial hyperenhancement,
washout). Nếu trọng số gần đều 1/8 thì attention chưa học được gì.

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.train.loop import load_checkpoint

# load_checkpoint trả về payload dict, không tự nạp vào model.
# best.pt = {"model": state_dict, "epoch": ..., "metrics": ..., "fold": ...}
payload = load_checkpoint(RUN_DIR / "best.pt")
best_model = build_model(CFG["model"]).to(DEVICE)
best_model.load_state_dict(payload["model"])
best_model.eval()
print(f"nạp best.pt @ epoch {payload['epoch']}")

phases = [p["name"] for p in load_yaml(REPO / "configs" / "data.yaml")["phases"]]
totals = torch.zeros(len(phases), device=DEVICE)
n = 0
with torch.no_grad():
    for batch in DataLoader(val_ds, batch_size=4):
        best_model(batch["image"].to(DEVICE))
        w = best_model.last_phase_weights
        if w is None:
            print("fusion không phải 'attention' — không có trọng số để xem")
            break
        totals += w.sum(0)
        n += len(w)

if n:
    mean_w = (totals / n).cpu()
    print(f"trọng số attention trung bình trên {n} ca val (đều = {1 / len(phases):.4f}):\n")
    for name, w in sorted(zip(phases, mean_w.tolist()), key=lambda kv: -kv[1]):
        bar = "#" * int(round(w * 200))
        print(f"   {name:>10}: {w:.4f}  {bar}")

## 4. Giữ lại gì

**Bắt buộc, vài trăm KB** — gói tải về máy:
`val_probs_best.npz` · `metrics_best.json` · `train_log.csv` · `config_used.json`

`val_probs_best.npz` là file quan trọng nhất: toàn bộ calibration và selective tính
lại được từ nó, trên CPU, không cần checkpoint.

**`best.pt` (~46 MB)** — Save Version để giữ. Cần cho Grad-CAM, web app, test-104, và
để làm một thành viên của deep ensemble sau này.

**`last.pt` (~131 MB)** — chỉ giữ nếu run **chưa** chạy đủ 300 epoch, để còn resume.

In [ ]:
import shutil
from pathlib import Path

out = Path(f"/kaggle/working/{EXPERIMENT}_results")
out.mkdir(exist_ok=True)
for name in ["val_probs_best.npz", "metrics_best.json", "train_log.csv", "config_used.json"]:
    shutil.copy(RUN_DIR / name, out / name)
shutil.make_archive(str(out), "zip", out)
print("đã gói:", round(Path(f"{out}.zip").stat().st_size / 2**10, 1), "KiB")

for f in sorted(RUN_DIR.iterdir()):
    print(f"   {f.name:24s} {f.stat().st_size / 2**20:8.1f} MiB")

from IPython.display import FileLink

FileLink(f"{EXPERIMENT}_results.zip")